# Customer Data Analysis - Data Cleaning
### Step 2: Data Cleaning using Python
This notebook cleans the customer shopping dataset so it is ready for SQL and Power BI.

---
## Cell 1 — Import Libraries and Load Data
**Why?** Before we do anything, we need to bring in our tools (libraries) and load the dataset into Python.

In [ ]:
import pandas as pd      # pandas helps us work with table data (like Excel)
import numpy as np       # numpy helps with numbers and math

# Load the CSV file into a DataFrame called 'df'
# A DataFrame is just a table with rows and columns — like Excel inside Python
df = pd.read_csv('Customer-ProjectData.csv')

# df.shape tells us (number of rows, number of columns)
print("Dataset size (rows, columns):", df.shape)

---
## Cell 2 — Preview the Data
**Why?** We look at the first 5 rows to confirm the data loaded correctly and understand what the columns look like.

In [ ]:
# df.head() shows the first 5 rows of the table
# This helps us see column names, sample values, and the structure of the data
df.head()

---
## Cell 3 — Check Column Info and Data Types
**Why?** df.info() shows us two important things:
- The data type of each column (is it a number, text, decimal?)
- How many non-null (non-empty) values each column has — so we can spot missing values

In [ ]:
# df.info() prints a summary of every column
# Look at the 'Non-Null Count' column — if it's less than 3900, that column has missing values
# Look at 'Dtype' — object means text, int64 means whole number, float64 means decimal number
df.info()

---
## Cell 4 — Check Statistics for Number Columns
**Why?** df.describe() shows us min, max, average, and more for all number columns.
This helps us spot invalid values — for example, if Age has a minimum of -5 or Review Rating has a max of 99, those are errors we need to fix.

In [ ]:
# df.describe() works only on number columns (int and float)
# count = how many values exist | mean = average | min = smallest | max = largest
df.describe()

---
## Cell 5 — Count Missing Values
**Why?** We need to know exactly which columns have missing (empty) values and how many, so we can fix them.

In [ ]:
# df.isnull() checks every cell — True if empty, False if it has a value
# .sum() counts the True values per column
# Result: columns with 0 are perfect, columns with a number have that many missing values
print("Missing values in each column:")
print(df.isnull().sum())

---
## Cell 6 — Count Duplicate Rows
**Why?** If the same customer record appears more than once, it will be counted twice in our analysis — giving us wrong totals and averages.

In [ ]:
# df.duplicated() checks if any row is an exact copy of another row
# .sum() counts how many duplicates exist
# If the result is 0, we have no duplicates — great!
print("Number of duplicate rows:", df.duplicated().sum())

---
## Cell 7 — Fix Missing Values in Review Rating
**Why?** Review Rating has 37 missing values (3900 - 3863 = 37). We fill them with the median (middle value) of all ratings.
We use median instead of average/mean because median is not affected by extreme values — it stays realistic.

> **Note:** We write `df['Review Rating'] = df['Review Rating'].fillna(...)` instead of using `inplace=True`.
> The `inplace=True` way gives a warning in newer versions of pandas and may not work correctly.

In [ ]:
# First, let's see what the median rating is
print("Median Review Rating:", df['Review Rating'].median())

# fillna() fills all empty cells with the value we choose (the median here)
# We assign the result back to df['Review Rating'] to save the change
df['Review Rating'] = df['Review Rating'].fillna(df['Review Rating'].median())

# Confirm — Review Rating should now show 0 missing values
print("\nMissing values after fix:")
print(df.isnull().sum())

---
## Cell 8 — Remove Duplicate Rows
**Why?** Even though we found 0 duplicates, it is good practice to always run this step. If there were duplicates, this removes them.

In [ ]:
# drop_duplicates() finds all rows that are exact copies and removes them
# It keeps the first occurrence and deletes the rest
df = df.drop_duplicates()

# len(df) counts the number of rows — we print it to confirm how many rows remain
print("Rows after removing duplicates:", len(df))

---
## Cell 9 — Check for Inconsistent Values in Text Columns
**Why?** Sometimes the same value is written differently — like 'Male' and 'male' or 'Yes' and 'YES'.
If we don't fix this, Power BI will show them as two separate groups — splitting the data incorrectly.

In [ ]:
# .unique() returns every distinct value found in a column
# We use this to visually check for typos or inconsistencies

print("Gender values:      ", df['Gender'].unique())
print("Season values:      ", df['Season'].unique())
print("Category values:    ", df['Category'].unique())
print("Size values:        ", df['Size'].unique())
print("Subscription values:", df['Subscription Status'].unique())

---
## Cell 10 — Fix Data Types
**Why?** Python sometimes reads columns with the wrong type. For example:
- Customer ID is read as a number, but it is an ID label — it should be text (string)
- Purchase Amount should be a decimal (float) so we can do math on it properly

In [ ]:
# astype(str)   → converts to text (string) — good for IDs, labels
# astype(float) → converts to decimal number — good for money, ratings
# astype(int)   → converts to whole number — good for age, counts

df['Customer ID'] = df['Customer ID'].astype(str)               # ID is a label, not a number
df['Purchase Amount (USD)'] = df['Purchase Amount (USD)'].astype(float)  # needs decimals for math
df['Age'] = df['Age'].astype(int)                               # age is always a whole number

# Print all column types to confirm the changes
print("Column data types after fixing:")
print(df.dtypes)

---
## Cell 11 — Rename Columns (Clean Names for SQL)
**Why?** SQL and Power BI do not work well with:
- Spaces in column names (e.g. 'Purchase Amount')
- Special characters like ( and )
- UPPERCASE names (harder to type in SQL)

So we convert all column names to lowercase, replace spaces with underscores, and rename the tricky column.

In [ ]:
# str.lower() → makes all column names lowercase: 'Gender' becomes 'gender'
df.columns = df.columns.str.lower()

# str.replace(' ', '_') → replaces spaces with underscores: 'item purchased' becomes 'item_purchased'
df.columns = df.columns.str.replace(' ', '_')

# rename() → renames one specific column with a tricky name
# 'purchase_amount_(usd)' has brackets which cause errors in SQL
# We rename it to the simple clean name 'purchase_amount'
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

# Print the final list of column names to confirm everything looks correct
print("Final clean column names:")
print(df.columns.tolist())

---
## Cell 12 — Final Check Before Saving
**Why?** Before saving, we do one last check to confirm:
- No missing values remain
- The data looks correct
- Column names are clean

In [ ]:
# Final summary check
print("=== FINAL DATA SUMMARY ===")
print(f"Total rows    : {len(df)}")
print(f"Total columns : {len(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicates    : {df.duplicated().sum()}")
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 3 rows preview:")
df.head(3)

---
## Cell 13 — Save the Cleaned Dataset
**Why?** All our cleaning work only exists in Python's memory right now.
We must save it as a CSV file so we can import it into MySQL (Step 3) and Power BI (Step 4).
Without this step, all our work is lost when we close Jupyter!

In [ ]:
# to_csv() saves the DataFrame as a CSV file
# 'customer_data_analysis.csv' is the file name — it will be created in your project folder
# index=False means do NOT add an extra row-number column (0,1,2,3...) — we don't need it
df.to_csv('customer_data_analysis.csv', index=False)

print("✅ Cleaned dataset saved as: customer_data_analysis.csv")
print("You can now import this file into MySQL and Power BI!")